# Cell 9: Probe Robustness Check
This notebook tests if the Layer 13 causal result is stable across different probe training runs (bootstrapping and regularization). 
**Requirements:** Attach your `features` dataset (the unzipped `probe_features.npz` files) to this notebook before running.

In [ ]:
!pip install -q huggingface_hub scikit-learn numpy torch

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

print('Downloading hidden states from HuggingFace (this may take a few minutes)...')
hs_local = Path(snapshot_download('AnishRacherla/LinguaFranca-Phase3', repo_type='dataset', allow_patterns='data/hidden_states/*'))
HS_DIR = hs_local / 'data' / 'hidden_states'
print(f'Hidden states ready at {HS_DIR}')

In [ ]:
import json
import os
import numpy as np
import torch
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from huggingface_hub import hf_hub_download

# Find the unzipped numpy arrays in Kaggle
FEATURES_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'ids.npy' in files and 'F_clean.npy' in files:
        FEATURES_DIR = root
        break
if not FEATURES_DIR:
    raise FileNotFoundError("Could not find the unzipped .npy files in /kaggle/input/.")

def load_npy(name):
    return np.load(os.path.join(FEATURES_DIR, name + '.npy'), allow_pickle=True)

REPO = "AnishRacherla/LinguaFranca-Phase3"
LAYERS = [9, 10, 11, 12, 13]
N_BOOT = 5                     # bootstrap resamples at the default C
C_VALUES = [1.0, 0.1, 0.01]    # regularisation strengths

labels_file = hf_hub_download(REPO, "data/2wikimultihopqa/augmented.jsonl", repo_type="dataset")
train_ids = set(map(str, json.load(open(hf_hub_download(REPO, "probes/probe_train_ids.json", repo_type="dataset")))))
examples = [json.loads(l) for l in open(labels_file)]

X = {L: [] for L in LAYERS}; y, groups = [], []
for ex in examples:
    if str(ex["id"]) not in train_ids:
        continue
    p = HS_DIR / f"{ex['id']}.pt"
    if not p.exists():
        continue
    pooled = torch.load(p, map_location="cpu", weights_only=False)["pooled"]
    for hop in ex.get("hops", []):
        if hop["hop_idx"] == 1 and hop.get("label") in (0, 1) and pooled.shape[1] > 0:
            for L in LAYERS:
                X[L].append(pooled[L, 0].float().numpy())
            y.append(hop["label"]); groups.append(str(ex.get("clean_pair_id", ex["id"])))
y = np.array(y); groups = np.array(groups)
X = {L: np.nan_to_num(np.stack(v)) for L, v in X.items()}
print(f"Hop-1 training set: {len(y)} hops, failure rate {y.mean():.3f}")

FL = list(load_npy('feature_layers'))
ids = load_npy('ids').astype(str)
held = np.array([i not in train_ids for i in ids])

F_clean = load_npy('F_clean')
F_cf = load_npy('F_cf')
F_entity = load_npy('F_entity')

def fit(L, C, idx):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=0))
    return clf.fit(X[L][idx], y[idx])

def evaluate(clf, L):
    k = FL.index(L)
    s = lambda A: clf.decision_function(np.nan_to_num(A.reshape(-1, A.shape[-1]).astype(np.float64))).reshape(A.shape[:-1])
    cl, co, ent = s(F_clean[:, k]), s(F_cf[:, k]), s(F_entity[:, :, k])
    d = co - cl
    frac = ((co[:, None] - ent[:, :L]).mean(0) / (co - cl).mean()).max()
    return d.mean(), d[held].mean(), (d > 0).mean(), frac

rng = np.random.default_rng(0)
uniq = np.unique(groups)
print(f"\n{'L':>3} | {'variant':<14} | TestA all | TestA held | %rise | best restore frac")
summary = {}
for L in LAYERS:
    res = []
    for C in C_VALUES:
        res.append((f"C={C}", *evaluate(fit(L, C, np.arange(len(y))), L)))
    for b in range(N_BOOT):
        g = rng.choice(uniq, len(uniq), replace=True)
        idx = np.concatenate([np.where(groups == gi)[0] for gi in g])
        res.append((f"boot {b}", *evaluate(fit(L, 1.0, idx), L)))
    for name, a, h, pr, fr in res:
        print(f"{L:>3} | {name:<14} | {a:+9.2f} | {h:+10.2f} | {pr*100:4.0f}% | {fr:+.2f}")
    signs = np.sign([r[1] for r in res])
    verdict = ("STABLE POSITIVE (passes)" if (signs > 0).all() else
               "STABLE NEGATIVE (fails)" if (signs < 0).all() else "UNSTABLE - depends on the probe, don't claim")
    summary[L] = verdict
    print(f"    -> layer {L}: Test A sign across {len(res)} retrainings: {verdict}\n")

print("SUMMARY:", json.dumps(summary, indent=1))
